# Claude 图片理解（Image Understanding）— Anthropic SDK

本 Notebook 演示如何通过 **七牛 AIToken** 平台，使用 **Anthropic Python SDK** 直接调用 Claude Messages API，以 Base64 编码方式传入本地图片进行多模态理解。

## 功能简介

与 Claude Agent SDK 的 Read 工具方式不同，本示例直接使用 Anthropic Messages API：
- **手动 Base64 编码**：读取本地图片文件，编码为 base64 字符串
- **Messages API 直接调用**：通过 `client.messages.create()` 传入图片内容块
- **无需 Agent 框架**：适用于轻量级图片理解场景，无需 Agent SDK 的工具调用机制

**API 端点**：`https://api.qnaigc.com`  
**适用模型**：`claude-4.6-sonnet`

## 1. 环境准备

安装 `anthropic` 依赖包，并设置 API Key。

In [ ]:
# 安装 Anthropic Python SDK
%pip install anthropic -q

In [ ]:
import os
import base64
from pathlib import Path
from anthropic import Anthropic

# 七牛 AIToken 平台地址
BASE_URL = "https://api.qnaigc.com"

# 从环境变量读取 API Key（或替换为你的 API Key）
API_KEY = os.environ.get("QINIU_API_KEY", "<your-api-key>")

# 使用的模型
MODEL = "claude-4.6-sonnet"

# 初始化 Anthropic 客户端，指向七牛 AIToken 平台
client = Anthropic(
    api_key=API_KEY,
    base_url=BASE_URL,
)

print("环境配置完成!")
print(f"  API 端点: {BASE_URL}")
print(f"  模型: {MODEL}")

## 2. 读取并编码图片

读取本地图片文件，将其编码为 Base64 字符串，用于后续 API 调用。

支持的图片格式：
- JPEG (`*.jpg`, `*.jpeg`)
- PNG (`*.png`)
- GIF (`*.gif`)
- WebP (`*.webp`)

In [ ]:
# 图片路径（与本 Notebook 同目录下的 image.png）
IMAGE_PATH = Path("image.png")
assert IMAGE_PATH.exists(), f"图片不存在: {IMAGE_PATH}"

# 读取图片文件并编码为 Base64
image_data = base64.standard_b64encode(IMAGE_PATH.read_bytes()).decode("utf-8")

# 根据文件扩展名确定 MIME 类型
MEDIA_TYPE = "image/png"

print(f"图片路径: {IMAGE_PATH}")
print(f"图片大小: {IMAGE_PATH.stat().st_size / 1024:.1f} KB")
print(f"媒体类型: {MEDIA_TYPE}")
print(f"Base64 编码长度: {len(image_data)} 字符")

## 3. 图片理解

使用 `client.messages.create()` 调用 Claude Messages API，在请求体中以 Base64 格式传入图片。

请求体中图片内容块的格式：
```json
{
    "type": "image",
    "source": {
        "type": "base64",
        "media_type": "image/png",
        "data": "<base64_encoded_data>"
    }
}
```

In [ ]:
# 调用 Claude Messages API，传入 Base64 编码的图片
message = client.messages.create(
    model=MODEL,
    max_tokens=1024,
    messages=[
        {
            "role": "user",
            "content": [
                {
                    "type": "image",
                    "source": {
                        "type": "base64",
                        "media_type": MEDIA_TYPE,
                        "data": image_data,
                    },
                },
                {
                    "type": "text",
                    "text": "请详细描述这张图片的内容，包括主要元素、颜色、构图和整体氛围。",
                },
            ],
        }
    ],
)

# 打印模型回复
print("=== 模型回复 ===")
print(message.content[0].text)

# 打印 Token 用量信息
print(f"\n--- 用量信息 ---")
print(f"模型: {message.model}")
print(f"停止原因: {message.stop_reason}")
print(f"输入 Tokens: {message.usage.input_tokens}")
print(f"输出 Tokens: {message.usage.output_tokens}")

## 4. 自定义提示词

可以通过修改文本提示词，让 Claude 从不同角度分析图片。

In [ ]:
# 使用不同的提示词分析同一张图片
message = client.messages.create(
    model=MODEL,
    max_tokens=1024,
    messages=[
        {
            "role": "user",
            "content": [
                {
                    "type": "image",
                    "source": {
                        "type": "base64",
                        "media_type": MEDIA_TYPE,
                        "data": image_data,
                    },
                },
                {
                    "type": "text",
                    "text": "Please analyze this image from a photography perspective. Comment on the composition, lighting, color palette, and artistic techniques used.",
                },
            ],
        }
    ],
)

print("=== Model Response ===")
print(message.content[0].text)

print(f"\n--- Usage ---")
print(f"Input tokens: {message.usage.input_tokens}")
print(f"Output tokens: {message.usage.output_tokens}")

## 5. 参数说明

### Anthropic 客户端初始化

| 参数 | 类型 | 说明 |
|------|------|------|
| `api_key` | string | API Key，使用七牛 AIToken 的 API Key |
| `base_url` | string | API 端点地址，设为 `https://api.qnaigc.com` 指向七牛 AIToken 平台 |

### messages.create() 主要参数

| 参数 | 类型 | 必填 | 说明 |
|------|------|------|------|
| `model` | string | 是 | 模型 ID，如 `claude-4.6-sonnet` |
| `max_tokens` | integer | 是 | 最大输出 Token 数 |
| `messages` | array | 是 | 消息列表，包含 `role` 和 `content` |
| `system` | string | 否 | 系统提示词 |
| `temperature` | float | 否 | 采样温度，范围 0~1，默认 1 |

### 图片内容块格式

| 字段 | 类型 | 说明 |
|------|------|------|
| `type` | string | 固定为 `"image"` |
| `source.type` | string | 固定为 `"base64"` |
| `source.media_type` | string | MIME 类型：`image/png`、`image/jpeg`、`image/gif`、`image/webp` |
| `source.data` | string | Base64 编码的图片数据 |

### 响应对象

| 字段 | 说明 |
|------|------|
| `message.content[0].text` | 模型的文本回复 |
| `message.model` | 实际使用的模型 ID |
| `message.stop_reason` | 停止原因（`end_turn`、`max_tokens` 等） |
| `message.usage.input_tokens` | 输入 Token 数（含图片 Token） |
| `message.usage.output_tokens` | 输出 Token 数 |

## 6. 与 Agent SDK 方式的对比

| 特性 | Anthropic SDK（本示例） | Claude Agent SDK |
|------|------------------------|------------------|
| 图片传入方式 | 手动 Base64 编码，放入请求体 | Agent 自动调用 Read 工具读取文件 |
| 依赖包 | `anthropic` | `claude-agent-sdk` |
| 调用方式 | `client.messages.create()` 同步调用 | `query()` 异步迭代器 |
| 灵活性 | 完全控制请求参数 | Agent 框架自动管理 |
| 适用场景 | 轻量级、精确控制 | 需要工具调用、多轮 Agent 交互 |

## 更多资源

- [七牛 AIToken 文档](https://developer.qiniu.com/aitokenapi)
- [Anthropic Python SDK 文档](https://github.com/anthropics/anthropic-sdk-python)
- [Anthropic Messages API 参考](https://docs.anthropic.com/en/api/messages)
- [七牛 API 调用示例文档](https://apidocs.qnaigc.com)